<a href="https://colab.research.google.com/github/RafaelSeri/TP_LABO_MET/blob/main/notebooks/Tp_labo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabajo Práctico Grupal — Laboratorio de Métodos Cuantitativos
### FCE UBA · 2do Cuatrimestre 2026

**Integrantes:** Catalina Fiorini · Rafael Seri · Federico Mingorance · Micaela Berzoni · Genaro D'Angelo

**Repositorio:** [github.com/RafaelSeri/TP_LABO_MET](https://github.com/RafaelSeri/TP_LABO_MET)

---

## ¿Por qué la demanda eléctrica?

Cada verano aparecen las mismas noticias: récords de consumo, cortes de luz y redes al límite. Cada invierno, la discusión pasa al gas y a la calefacción. Detrás de esos titulares hay una variable que nadie controla: **la temperatura**.

En este trabajo analizamos la demanda de energía eléctrica en Argentina entre 2005 y 2025 para entender cuánto depende del clima y en qué momentos del año el sistema está más exigido. Esta pregunta no es solo curiosidad: planificar la generación, anticipar picos de consumo y diseñar tarifas depende de saber cómo responde la demanda cuando hace calor o frío.

## Los datos

La base reúne información de **[FUENTE — Ministerio de Economía]** con: http://argentina.gob.ar/economia

- **Demanda total** y su desagregación en tres segmentos: residencial, comercio e industria, y grandes usuarios.
- **Temperatura promedio** del período.
- **Potencia máxima** registrada.
- **Demanda por empresa distribuidora** (EDENOR, EDESUR, EPEC, etc.), disponible desde 2014.

Los datos tienen tres frecuencias: anual, trimestral y mensual. Teniendo en cuenta las estaciones del año, la serie mensual es la que más información aporta sobre la relación con la temperatura.

## Pregunta que guía el trabajo

> **¿Cuánto influye la temperatura en la demanda eléctrica en Argentina y en qué épocas del año resulta más crítica?**
## Estructura del notebook

1. Carga y exploración inicial de los datos
2. Definición de la pregunta de investigación
3. Transformaciones y resúmenes estadísticos
4. Análisis gráfico
5. Respuesta a la pregunta y conclusiones

# 1. Carga y exploración inicial de los datos.
###1.1 Carga de datos

En esta sección cargamos la base de datos y la recorremos antes de transformarla: qué columnas tiene, de qué tipo son, qué período cubre y si hay valores faltantes o duplicados. El objetivo es conocer los datos antes de sacar conclusiones.

In [ ]:
import pandas as pd
import numpy as np

ruta = 'https://github.com/RafaelSeri/TP_LABO_MET/blob/main/DF/base_limpia.xlsx?raw=1'
df = pd.read_excel(ruta)

# Miramos cómo viene escrito el período en distintas partes de la tabla
df['Período'].iloc[[0, 20, 21, 22, 106, 107, 366]]

,Período
0,2005
20,2025
21,I 05
22,II
106,II
107,2005-01-01 00:00:00
366,2026-08-01 00:00:00


### 1.2 Identificación de la frecuencia de cada fila

La base tiene datos con tres frecuencias distintas apilados en una misma tabla: primero los anuales, después los trimestrales y por último los mensuales. Cada uno tiene su propio formato en la columna `Período`:

- **Anual:** el año como número (`2005`).
- **Trimestral:** en números romanos; solo el primer trimestre indica el año (`I 05`, `II`, `III`, `IV`).
- **Mensual:** una fecha completa (`2005-01-01`).

Creamos la columna `Frecuencia` a partir de ese formato. Es más seguro que separar por número de fila: si la base se modifica, la clasificación sigue funcionando.

In [ ]:
def clasificar_frecuencia(p):
    if isinstance(p, str):                  # 'I 05', 'II', 'III', 'IV'
        return 'trimestral'
    if isinstance(p, (int, np.integer)):    # 2005, 2006, ...
        return 'anual'
    return 'mensual'                        # lo que queda son las fechas

df['Frecuencia'] = df['Período'].apply(clasificar_frecuencia)

# Verificación: esperamos 21 anuales, 86 trimestrales y 260 mensuales.
df['Frecuencia'].value_counts()

,count
Frecuencia,
mensual,260
trimestral,86
anual,21


### 1.3 Separación en dos tablas

Las columnas de la base responden a dos temas distintos, así que las separamos:

- **`df_base`**: demanda total, demanda por segmento (residencial, comercio e industria, grandes usuarios), temperatura promedio y potencia máxima. Es la tabla central para responder nuestra pregunta.
- **`df_empresas`**: demanda de cada empresa distribuidora.

Usamos `.copy()` para que cada tabla sea independiente y las modificaciones posteriores no afecten la base original.

In [ ]:
#Transformamos la base limpia en dataframe y creamos dos subdataframes
df_base=df[['Período','Frecuencia','Demanda Total', 'Demanda Residencial',   # <- agregado 'Frecuencia'
            'Comercio e Industria','Grandes Usuarios','Temperatura Promedio',
            'Potencia Máxima']].copy()                                       # <- agregado .copy()
df_empresas=df[['Período','Frecuencia','EDELAP SA', 'EDENOR DISTRIBUIDOR',   # <- agregado 'Frecuencia'
'EDESAL DISTRIBUIDOR', 'EDESTESA(EMP.DIST.EL.DEL ESTE)',
'EDESUR DISTRIBUIDOR', 'EMP DE ENERGIA DE RIO NEGRO SA',
'EMP DIST ENERG ATLANTICA', 'EMP. DE ENERGIA DE LA RIOJA SA',
'EMP.DIST.ENERGIA DE SALTA', 'EMP.ELECTRIC.DE MISIONES S.A.',
'EMPRESA DIS. S. ESTERO SA', 'EMPRESA JUJENIA DE ENERGIA SA',
'ENERGIA DE CATAMARCA S.A.', 'ENERGIA DE ENTRE RIOS SA',
'ENERGIA DE MENDOZA SA', 'ENERGIA SAN JUAN SA EX-EDESSA',
'EPEC DISTRIBUIDOR', 'EPEN DISTRIBUIDOR', 'EPESF DISTRIBUIDOR',
'RECURSOS Y ENERGIA FORMOSA SA', 'SECHEEP', 'SPSE SANTA CRUZ']].copy()       # <- agregado .copy()
df_empresas=df_empresas.dropna() #nos quedamos con las filas con datos no nulos
df_empresas=df_empresas.reset_index(drop=True) #reseteo índices

### 1.4 Separación por frecuencia

Para cada tabla creamos tres subconjuntos según la columna `Frecuencia`: anual, trimestral y mensual. Filtrar por esa columna, en lugar de por número de fila, hace que la separación sea correcta aunque la base cambie o se eliminen filas con nulos.

Para responder nuestra pregunta vamos a trabajar principalmente con los **datos mensuales**: la temperatura promedio anual casi no varía de un año a otro, mientras que la mensual muestra con claridad el paso de las estaciones.

In [ ]:
# Para cada df, nos quedamos con los datos del período anual, trimestral y mensual
df_base_anual       = df_base[df_base['Frecuencia'] == 'anual'].copy()
df_base_trimestral  = df_base[df_base['Frecuencia'] == 'trimestral'].copy()
df_base_mensual     = df_base[df_base['Frecuencia'] == 'mensual'].copy()

df_empresas_anual      = df_empresas[df_empresas['Frecuencia'] == 'anual'].copy()
df_empresas_trimestral = df_empresas[df_empresas['Frecuencia'] == 'trimestral'].copy()
df_empresas_mensual    = df_empresas[df_empresas['Frecuencia'] == 'mensual'].copy()


### 1.5 Primeras y últimas filas

Revisamos el principio y el final de cada tabla. En la tabla base se ve la estructura apilada: las primeras filas son totales anuales y las últimas son datos mensuales, con valores de un orden de magnitud menor.

In [ ]:
df_base.head()

,Período,Frecuencia,Demanda Total,Demanda Residencial,Comercio e Industria,Grandes Usuarios,Temperatura Promedio,Potencia Máxima
0,2005,anual,92332.95,30086.495,43102.875,19143.580,18.363667,16143.0
1,2006,anual,96812.80,32608.358,41291.755,22912.687,18.575000,17395.0
2,2007,anual,102950.30,37111.179,43484.381,22354.740,17.650000,18345.0
3,2008,anual,105958.10,38610.044,43060.486,24287.570,19.050000,19126.0
4,2009,anual,104622.40,39564.998,42648.252,22409.150,18.583333,19566.0


In [ ]:
df_empresas.head()

,Período,Frecuencia,EDELAP SA,EDENOR DISTRIBUIDOR,EDESAL DISTRIBUIDOR,EDESTESA(EMP.DIST.EL.DEL ESTE),EDESUR DISTRIBUIDOR,EMP DE ENERGIA DE RIO NEGRO SA,EMP DIST ENERG ATLANTICA,EMP. DE ENERGIA DE LA RIOJA SA,...,ENERGIA DE CATAMARCA S.A.,ENERGIA DE ENTRE RIOS SA,ENERGIA DE MENDOZA SA,ENERGIA SAN JUAN SA EX-EDESSA,EPEC DISTRIBUIDOR,EPEN DISTRIBUIDOR,EPESF DISTRIBUIDOR,RECURSOS Y ENERGIA FORMOSA SA,SECHEEP,SPSE SANTA CRUZ
0,2014,anual,2378.511734,20695.281360,1193.612775,746.308529,17088.832858,1263.940622,2626.465450,1115.073941,...,917.436808,2582.879226,3381.433381,1586.920665,8626.852806,871.880839,9288.999704,1179.717698,2435.771533,509.829498
1,2015,anual,2545.617350,22207.382744,1236.492585,757.037308,18064.754754,1364.005442,2773.577417,1164.582836,...,950.285160,2763.273181,3462.140383,1719.146438,8933.013610,946.857694,9720.505696,1227.170328,2592.745982,561.240525
2,2016,anual,2816.603973,24457.502373,1463.685001,748.361903,19477.180554,1420.263874,2807.465483,1249.020304,...,1054.115441,2925.621791,3403.144289,1942.065602,9298.827150,955.509350,10190.968190,1306.644835,2728.039692,576.744343
3,2017,anual,2599.359071,22013.499165,1309.465478,786.936648,17565.433992,1312.805676,2705.760206,1312.898964,...,1002.080749,2821.614678,3361.309857,1873.970624,9121.949346,916.490891,9775.564765,1302.460587,2786.641335,598.480563
4,2018,anual,2669.383878,22229.026813,1270.556777,766.126704,17641.587070,1338.813501,2725.457446,1298.939775,...,1005.302807,2863.528905,3219.126411,1838.278127,9270.803419,905.697463,9860.999109,1353.646156,3005.021072,634.323649


In [ ]:
df_base.tail(10)

,Período,Frecuencia,Demanda Total,Demanda Residencial,Comercio e Industria,Grandes Usuarios,Temperatura Promedio,Potencia Máxima
362,2026-04-01 00:00:00,mensual,10579.342832,4463.562615,3969.723032,2146.057185,19.7,23251.0
363,2026-05-01 00:00:00,mensual,12151.531183,5825.605102,4075.780308,2250.145773,13.5,24345.0
364,2026-06-01 00:00:00,mensual,12641.171137,6339.790452,4114.308993,2187.071692,12.0,25785.0
365,2026-07-01 00:00:00,mensual,13063.214629,6643.194389,4211.154804,2208.865436,12.5,27189.0
366,2026-08-01 00:00:00,mensual,12432.027715,6093.065132,4100.504724,2238.457859,12.3,24956.0


In [ ]:
df_base.tail()

,Período,Frecuencia,Demanda Total,Demanda Residencial,Comercio e Industria,Grandes Usuarios,Temperatura Promedio,Potencia Máxima
362,2026-04-01 00:00:00,mensual,10579.342832,4463.562615,3969.723032,2146.057185,19.7,23251.0
363,2026-05-01 00:00:00,mensual,12151.531183,5825.605102,4075.780308,2250.145773,13.5,24345.0
364,2026-06-01 00:00:00,mensual,12641.171137,6339.790452,4114.308993,2187.071692,12.0,25785.0
365,2026-07-01 00:00:00,mensual,13063.214629,6643.194389,4211.154804,2208.865436,12.5,27189.0
366,2026-08-01 00:00:00,mensual,12432.027715,6093.065132,4100.504724,2238.457859,12.3,24956.0


### 1.6 Estructura de la base

Revisamos las dimensiones de la base y el tipo de dato de cada columna, para detectar columnas que pandas haya interpretado de forma distinta a la esperada.

In [ ]:
print('Filas y columnas de la base completa:', df.shape)
print('Filas y columnas de df_base:', df_base.shape)
print('Filas y columnas de df_empresas:', df_empresas.shape)
print()
df_base.dtypes

Filas y columnas de la base completa: (367, 30)
Filas y columnas de df_base: (367, 8)
Filas y columnas de df_empresas: (208, 24)



,0
Período,object
Frecuencia,object
Demanda Total,float64
Demanda Residencial,float64
Comercio e Industria,float64
Grandes Usuarios,float64
Temperatura Promedio,float64
Potencia Máxima,float64


### 1.7 Control de consistencia entre frecuencias

Verificamos que la suma de los trimestres y la suma de los meses de cada año coincidan con el dato anual. Calculamos la diferencia porcentual para cada año completo y mostramos la mayor diferencia encontrada en cada columna de demanda.

In [ ]:
anual      = df_base['Frecuencia'] == 'anual'
trimestral = df_base['Frecuencia'] == 'trimestral'
mensual    = df_base['Frecuencia'] == 'mensual'

def anio_trimestre(p):
    partes = str(p).split()          # 'I 05' -> ['I', '05'] ; 'II' -> ['II']
    if len(partes) == 2:
        return 2000 + int(partes[1])
    return np.nan                    # sin año: lo completamos después

df_base['Año'] = np.nan
df_base.loc[anual, 'Año'] = df_base.loc[anual, 'Período']
df_base.loc[mensual, 'Año'] = pd.to_datetime(df_base.loc[mensual, 'Período']).dt.year
df_base.loc[trimestral, 'Año'] = df_base.loc[trimestral, 'Período'].apply(anio_trimestre)
df_base.loc[trimestral, 'Año'] = df_base.loc[trimestral, 'Año'].ffill()

df_base['Año'] = df_base['Año'].astype(int)

# Verificación: cuántas filas quedaron por año en cada frecuencia
df_base.groupby(['Frecuencia', 'Año']).size().unstack(level=0).head()

/tmp/ipykernel_2140/1771112962.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[2005 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018
 2019 2020 2021 2022 2023 2024 2025]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_base.loc[anual, 'Año'] = df_base.loc[anual, 'Período']
/tmp/ipykernel_2140/1771112962.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_base.loc[trimestral, 'Año'] = df_base.loc[trimestral, 'Año'].ffill()


Frecuencia,anual,mensual,trimestral
Año,,,
2005,1.0,12.0,4.0
2006,1.0,12.0,4.0
2007,1.0,12.0,4.0
2008,1.0,12.0,4.0
2009,1.0,12.0,4.0


In [ ]:
cols_suma = ['Demanda Total', 'Demanda Residencial', 'Comercio e Industria', 'Grandes Usuarios']

dato_anual = df_base[anual].set_index('Año')[cols_suma]

# Sumamos por año y nos quedamos solo con los años completos
suma_trim = df_base[trimestral].groupby('Año')[cols_suma].sum()
filas_trim = df_base[trimestral].groupby('Año').size()
suma_trim = suma_trim[filas_trim == 4]

suma_mens = df_base[mensual].groupby('Año')[cols_suma].sum()
filas_mens = df_base[mensual].groupby('Año').size()
suma_mens = suma_mens[filas_mens == 12]

# Diferencia porcentual respecto del dato anual
dif_trim = ((suma_trim - dato_anual) / dato_anual * 100).dropna()
dif_mens = ((suma_mens - dato_anual) / dato_anual * 100).dropna()

print('Mayor diferencia (%) entre la suma de trimestres y el dato anual:')
print(dif_trim.abs().max().round(3))
print('\nMayor diferencia (%) entre la suma de meses y el dato anual:')
print(dif_mens.abs().max().round(3))

Mayor diferencia (%) entre la suma de trimestres y el dato anual:
Demanda Total           0.001
Demanda Residencial     0.045
Comercio e Industria    0.919
Grandes Usuarios        2.032
dtype: float64

Mayor diferencia (%) entre la suma de meses y el dato anual:
Demanda Total           0.001
Demanda Residencial     0.045
Comercio e Industria    0.919
Grandes Usuarios        2.032
dtype: float64


**Resultado del control de consistencia**

La demanda total es consistente entre las tres frecuencias: la suma de los trimestres y la de los meses coinciden con el dato anual, con diferencias menores al 0,001 % atribuibles a redondeo. Lo mismo ocurre con la demanda residencial (diferencia máxima de 0,05 %).

En cambio, aparecen diferencias en dos segmentos: hasta 0,9 % en comercio e industria y hasta 2 % en grandes usuarios. Como el total sí coincide, estas diferencias se compensan entre sí. La explicación más probable es que algunos usuarios estén clasificados en un segmento en los datos anuales y en otro en los mensuales y trimestrales.

Además, las diferencias son idénticas al comparar contra trimestres y contra meses, lo que indica que los datos mensuales y trimestrales son consistentes entre sí y la discrepancia está en la desagregación del dato anual.

**Decisión:** dado que el análisis se basa en los datos mensuales, esta diferencia no afecta las conclusiones sobre la demanda total. Al analizar los segmentos por separado, se usarán siempre datos mensuales, para no mezclar criterios de clasificación distintos.

In [ ]:
rango_base = df_base.groupby('Frecuencia')['Año'].agg(['min', 'max', 'count'])
rango_base.columns = ['Primer año', 'Último año', 'Cantidad de filas']

print('Rango temporal de la tabla base:')
display(rango_base)

print('Rango temporal de la tabla de empresas:')
display(df_empresas.groupby('Frecuencia')['Período'].agg(['first', 'last', 'count']))

Rango temporal de la tabla base:


,Primer año,Último año,Cantidad de filas
Frecuencia,,,
anual,2005,2025,21
mensual,2005,2026,260
trimestral,2005,2026,86


Rango temporal de la tabla de empresas:


,first,last,count
Frecuencia,,,
anual,2014,2025,12
mensual,2014-01-01 00:00:00,2026-06-01 00:00:00,150
trimestral,I 15,II,46


**Rango temporal**

- La tabla base cubre **2005 a 2025** en frecuencia anual y **2005 a 2026** en frecuencia trimestral y mensual. El año 2026 está incompleto: hay datos mensuales hasta agosto y trimestrales hasta el segundo trimestre.
- La demanda por empresa distribuidora tiene un rango más corto: **2014 a 2025** en datos anuales, **enero 2014 a junio 2026** en mensuales y **primer trimestre 2015 a segundo trimestre 2026** en trimestrales.
- Hay dos diferencias entre ambas tablas. Los meses de julio y agosto de 2026 no están en la tabla de empresas porque tenían valores faltantes, y los trimestres de 2014 no figuran en la fuente aunque ese año sí tiene datos anuales y mensuales.

Para el análisis principal usamos la tabla base con datos mensuales, que ofrece la serie más larga (260 meses). Al comparar empresas, el período queda limitado a 2014 en adelante.

### 1.9 Calidad de los datos

Revisamos si hay valores faltantes y registros duplicados. Para los nulos mostramos solo las columnas que tienen alguno.

In [ ]:
nulos = df.isnull().sum()
print('Columnas con valores nulos:')
print(nulos[nulos > 0])

print('\nFilas duplicadas:', df.duplicated().sum())
print('Períodos repetidos:', df_base.duplicated(subset=['Frecuencia', 'Año', 'Período']).sum())

Columnas con valores nulos:
EDELAP SA                         159
EDENOR DISTRIBUIDOR               159
EDESAL DISTRIBUIDOR               159
EDESTESA(EMP.DIST.EL.DEL ESTE)    159
EDESUR DISTRIBUIDOR               159
EMP DE ENERGIA DE RIO NEGRO SA    159
EMP DIST ENERG ATLANTICA          159
EMP. DE ENERGIA DE LA RIOJA SA    159
EMP.DIST.ENERGIA DE SALTA         159
EMP.ELECTRIC.DE MISIONES S.A.     159
EMPRESA DIS. S. ESTERO SA         159
EMPRESA JUJENIA DE ENERGIA SA     159
ENERGIA DE CATAMARCA S.A.         159
ENERGIA DE ENTRE RIOS SA          159
ENERGIA DE MENDOZA SA             159
ENERGIA SAN JUAN SA EX-EDESSA     159
EPEC DISTRIBUIDOR                 159
EPEN DISTRIBUIDOR                 159
EPESF DISTRIBUIDOR                159
RECURSOS Y ENERGIA FORMOSA SA     159
SECHEEP                           159
SPSE SANTA CRUZ                   159
dtype: int64

Filas duplicadas: 0
Períodos repetidos: 0


**Valores nulos y duplicados**

- La tabla base (demanda, segmentos, temperatura y potencia) **no tiene valores faltantes**.
- Las 22 columnas de empresas distribuidoras tienen exactamente **159 nulos cada una**, siempre en las mismas filas. No se trata de datos faltantes aislados, sino de períodos completos sin información por empresa: 2005–2013 en datos anuales y mensuales, 2005–2014 en trimestrales, y julio y agosto de 2026 en mensuales. Por eso, al eliminar esas filas con `dropna()` no se pierde información parcial.
- **No hay filas duplicadas** ni períodos repetidos dentro de una misma frecuencia.

Para las estadísticas descriptivas usamos solo los datos mensuales. Si aplicáramos `.describe()` a toda la tabla, mezclaríamos totales anuales (alrededor de 100.000) con valores mensuales (alrededor de 10.000), y la media, el mínimo y el desvío no tendrían sentido.

In [ ]:
df_base[df_base['Frecuencia'] == 'mensual'].drop(columns='Año').describe().round(1)

,Demanda Total,Demanda Residencial,Comercio e Industria,Grandes Usuarios,Temperatura Promedio,Potencia Máxima
count,260.0,260.0,260.0,260.0,260.0,260.0
mean,10317.9,4323.2,4008.4,1986.3,18.6,21242.7
std,1480.3,1122.3,365.1,165.6,4.9,3391.3
min,7213.1,2184.8,3261.5,1403.8,9.3,14552.0
25%,9102.2,3509.9,3727.8,1909.6,14.3,18436.2
50%,10346.7,4286.4,4012.1,1988.1,18.4,21049.5
75%,11358.3,5101.3,4250.3,2099.1,23.0,23542.5
max,13996.9,6927.4,5016.6,2333.8,27.1,30257.0


**Estadísticas descriptivas (datos mensuales)**

- La demanda total mensual promedio es de 10.318 [unidad], con valores entre 7.213 y 13.997. En promedio, el consumo residencial representa el 42 % del total, comercio e industria el 39 % y grandes usuarios el 19 %.
- **La demanda residencial es el segmento más variable:** su desvío estándar equivale a alrededor del 26 % de su promedio, y su valor máximo triplica al mínimo. En cambio, comercio e industria y grandes usuarios muestran un comportamiento mucho más estable, con desvíos cercanos al 9 % y al 8 % de su promedio.
- La temperatura promedio mensual varía entre 9,3 y 27,1 °C, con una media de 18,6 °C.
- La potencia máxima casi se duplica entre su valor mínimo y su valor máximo.

Esto sugiere que la demanda residencial es la más sensible a las condiciones climáticas. Sin embargo, su variabilidad combina dos efectos: el crecimiento de la demanda a lo largo de los años y las variaciones estacionales. En las secciones siguientes se buscará separar ambos efectos.

## 2. Pregunta de investigación

### ¿Cuál es la pregunta?

> **¿Cuánto influye la temperatura en la demanda eléctrica mensual de Argentina entre 2005 y 2026, en qué épocas del año resulta más crítica y qué segmento de usuarios es más sensible a ella?**

Para responderla, la desglosamos en tres preguntas más concretas:

1. ¿Cómo varía la demanda a lo largo del año y en qué meses alcanza sus picos?
2. ¿Qué relación hay entre la temperatura promedio mensual y la demanda? ¿Aumenta solo con el calor, o también con el frío?
3. ¿Qué segmento (residencial, comercio e industria o grandes usuarios) responde más a los cambios de temperatura?

### ¿Por qué vale la pena responderla?

La demanda eléctrica no es constante: el sistema tiene que estar preparado para abastecer los momentos de mayor consumo, aunque ocurran pocas veces al año. Saber cuánto depende la demanda del clima permite anticipar esos picos, planificar la generación y el mantenimiento de las redes, y entender por qué los cortes de luz se concentran en ciertas épocas.

En la exploración inicial ya apareció una pista: la demanda residencial es mucho más variable que la de comercio, industria y grandes usuarios. Esto sugiere que el efecto del clima pasa principalmente por el consumo de los hogares, con aires acondicionados en verano y calefacción eléctrica en invierno.

### ¿Qué respuesta encontramos?

COMPLETAR!!!!!!